## Retrieve activity data from Strava for storage and analysis

This notebook demonstrates how to retrieve activity data from Strava for storage and analysis. It uses the stravalib library to authenticate with Strava and retrieve activity data.


## 1. Connecting to the Supabase Database

In [10]:
import os
from dotenv import load_dotenv
import psycopg2
from stravalib import Client

# Load the keys out of your hidden .env file
load_dotenv()

# Safely extract the secret strings
client_id = os.getenv("STRAVA_CLIENT_ID")
client_secret = os.getenv("STRAVA_CLIENT_SECRET")

# Connect to Supabase using the hidden environmental variables
conn = psycopg2.connect(
    host=os.getenv("SUPABASE_HOST"),
    port="6543",
    user=os.getenv("SUPABASE_USER"),
    password=os.getenv("SUPABASE_PASSWORD"),
    database="postgres"
)
cursor = conn.cursor()

## 2. Authenticating with the Strava API

In [8]:
from stravalib import Client

# Set up Strava client
client = Client()

# Authenticate with Strava
client_id = client_id
client_secret = client_secret
redirect_uri = "http://localhost/"  # Must match the callback domain in your Strava app

# Generate the authorization URL
auth_url = client.authorization_url(
    client_id=int(client_id),
    redirect_uri=redirect_uri,
    scope="activity:read_all"  # Request read access to all activities
)

print(f"Go to this URL to authorize: {auth_url}")

# Get the authorization code from the URL
code = input("Enter the authorization code from the URL: ")

# Exchange the code for an access token
token = client.exchange_code_for_token(
    client_id=int(client_id),
    client_secret=client_secret,
    code=code
)

# Save the access token
TOKEN = token["access_token"]


Go to this URL to authorize: https://www.strava.com/oauth/authorize?client_id=267964&redirect_uri=http%3A%2F%2Flocalhost%2F&approval_prompt=auto&scope=activity%3Aread_all&response_type=code


SSLError: HTTPSConnectionPool(host='www.strava.com', port=443): Max retries exceeded with url: /oauth/token?client_id=267964&client_secret=31019d7c05cbb71314afe45e05a3cebdc64fbc31&code=905e365a695f7e61ffd58e9bde5098cf46d38f77&grant_type=authorization_code (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1077)')))

## 3. Collecting Athlete Data

In [ ]:
import requests

# Collecting athlete data from Strava API
athlete_url = "https://www.strava.com/api/v3/athlete?access_token=" + TOKEN
response_athlete = requests.get(athlete_url)
athlete_data = response_athlete.json()

# Extract athlete details
athlete_name = athlete_data["firstname"] + " " + athlete_data["lastname"]
gender = athlete_data["sex"]
age = athlete_data["age"] if "age" in athlete_data else None  # Check if age is available

# Verify if the athlete is already in the database
cursor.execute("SELECT id FROM athletes WHERE athlete_name = %s", (athlete_name,))
athlete = cursor.fetchone()

if athlete is None:
    # Insert new athlete
    cursor.execute("INSERT INTO athletes (athlete_name, gender, age) VALUES (%s, %s, %s)",
                   (athlete_name, gender, age))
    conn.commit()
    athlete_id = cursor.lastrowid  # Get the new athlete ID
else:
    athlete_id = athlete[0]  # Use the existing athlete ID

## 4. Collecting activity data

In [ ]:
activities_url = "https://www.strava.com/api/v3/athlete/activities?access_token=" + TOKEN
activities = []
page = 1
per_page = 150  # Number of activities per page

while True:
    # Fetch activities with pagination
    paginated_url = f"{activities_url}&page={page}&per_page={per_page}"
    response_activities = requests.get(paginated_url)
    data = response_activities.json()

    if not data:
        break  # Stop if no more activities are returned

    activities.extend(data)  # Add activities to the list
    page += 1  # Move to the next page

    time.sleep(1)  # Avoid hitting the API rate limit

## 5. Transforming the Data

In [ ]:
import pandas as pd
from datetime import datetime
import pytz

# Convert data to a DataFrame
df = pd.DataFrame(activities)

# Select relevant columns
df = df[['id', 'name', 'start_date', 'type', 'distance', 'moving_time',
         'total_elevation_gain', 'max_speed', 'average_speed']]

# Convert units
df["distance"] = df["distance"] / 1000  # Convert meters to kilometers
df["moving_time"] = df["moving_time"] / 60  # Convert seconds to minutes
df["max_speed"] = df["max_speed"] * 3.6  # Convert m/s to km/h
df["average_speed"] = df["average_speed"] * 3.6  # Convert m/s to km/h

# Adjust time zone and format for MySQL
from_zone = pytz.utc
to_zone = pytz.timezone("America/Sao_Paulo")  # Adjust to your time zone

def convert_to_mysql_datetime(start_date):
    start_date = start_date.rstrip('Z')
    dt = datetime.strptime(start_date, '%Y-%m-%dT%H:%M:%S')
    dt = from_zone.localize(dt).astimezone(to_zone)
    dt = dt.replace(tzinfo=None)  # Remove time zone info
    return dt

df["start_date"] = df["start_date"].apply(convert_to_mysql_datetime)

## 6. Loading data back into Supabase

In [ ]:
for _, row in df.iterrows():
    # Check if the activity already exists
    cursor.execute("SELECT id from activities WHERE activity_id = %s", (row["id"],))
    existing_activity = cursor.fetchone()

    if existing_activity is None:
        # Insert new activity
        cursor.execute("""
            INSERT INTO activities (athlete_id, activity_name, start_date, activity_type, distance, duration, elevation,
                                    max_speed, avg_speed, activity_id)
            VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
        """, (athlete_id, row["name"], row["start_date"], row["type"], row["distance"], row["moving_time"],
              row["total_elevation_gain"], row["max_speed"], row["average_speed"], row["id"]))
    else:
        # Update existing activity
        cursor.execute("""
            UPDATE activities
            SET athlete_id = %s, activity_name = %s, start_date = %s, activity_type = %s, distance = %s,
                duration = %s, elevation = %s, max_speed = %s, avg_speed = %s
            WHERE activity_id = %s
        """, (athlete_id, row["name"], row["start_date"], row["type"], row["distance"], row["moving_time"],
              row["total_elevation_gain"], row["max_speed"], row["average_speed"], row["id"]))

conn.commit()
cursor.close()
conn.close()
print("Athletes and activities inserted successfully!")